# Audiobook Enhancer — Colab Bootstrap

This notebook is a bootstrapper only. It never contains pipeline logic —
all processing code lives in the `audiobook_engine` package in the repo.
Edit code there, not here. See `02-Architecture.md` in the vault.

In [ ]:
# 1. Clone or pull the repo (absolute path — avoids nested-clone bugs from
# a lingering %cd in another cell's execution state)
REPO_DIR = "/content/audiobook-enhancer"
REPO_URL = "https://github.com/hasnaintypes/audiobook-enhancer.git"
import os
if os.path.isdir(REPO_DIR):
    !cd "$REPO_DIR" && git pull
else:
    !git clone "$REPO_URL" "$REPO_DIR"

In [ ]:
# 2. Install pinned dependencies
import os
os.chdir("/content/audiobook-enhancer")
!pip install -q -r config/requirements.lock
!pip install -q -e .

In [ ]:
# 3. Load Neon Object Storage credentials from Colab's Secrets manager
# (the key icon in the left sidebar) — NOT pasted into this cell. Add each
# of these as a secret there first, with "Notebook access" enabled:
#   AWS_ACCESS_KEY_ID, AWS_SECRET_ACCESS_KEY, AWS_ENDPOINT_URL_S3, AWS_REGION
# Values come from audiobook-enhancer/.env.local on your own machine
# (written there by `neon deploy` — see docs/07-Decisions.md D007).
import os
from google.colab import userdata

for key in ["AWS_ACCESS_KEY_ID", "AWS_SECRET_ACCESS_KEY", "AWS_ENDPOINT_URL_S3", "AWS_REGION"]:
    os.environ[key] = userdata.get(key)

print("Neon Object Storage credentials loaded from Colab Secrets.")

In [ ]:
# 4. Sanity check: run the fast unit test suite here to confirm the clone +
# dependency install actually produced a working environment.
!pytest tests/unit -q

In [ ]:
# 5. Example: run the full pipeline (extract -> analyze -> upload) against
# real sample URLs, writing originals to the Neon Object Storage bucket.
#
# extract_audio() in the engine always pulls the full video, so for a quick
# smoke test here we download only a CLIP_SECONDS-long window via yt-dlp's
# download_ranges instead of calling extract_audio() directly -- this is
# diagnostic-script logic for this cell only, not a change to the engine.
# The window is centered on the video's midpoint rather than the start.
script = '''
from pathlib import Path
import yt_dlp
from audiobook_engine.analysis.io import load_mono
from audiobook_engine.analysis.quality import analyze_quality
from audiobook_engine.analysis.strategy import select_strategy
from audiobook_engine.jobstore.storage import upload_file, object_key, presigned_url

URLS = [
    "https://youtu.be/XN9CVvZFkts",
]
CLIP_SECONDS = 60
OUTPUT_DIR = Path("/content/samples")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def extract_middle_clip(url, output_dir, clip_seconds):
    probe_opts = {"quiet": True, "no_warnings": True, "noplaylist": True}
    with yt_dlp.YoutubeDL(probe_opts) as ydl:
        probe_info = ydl.extract_info(url, download=False)
    duration = float(probe_info.get("duration") or 0.0)

    midpoint = duration / 2
    start = max(0.0, midpoint - clip_seconds / 2)
    end = min(duration, start + clip_seconds) if duration else start + clip_seconds

    ydl_opts = {
        "format": "bestaudio/best",
        "outtmpl": str(output_dir / "%(id)s.%(ext)s"),
        "postprocessors": [{"key": "FFmpegExtractAudio", "preferredcodec": "wav"}],
        "download_ranges": yt_dlp.utils.download_range_func(None, [(start, end)]),
        "force_keyframes_at_cuts": True,
        "noplaylist": True,
        "quiet": True,
        "no_warnings": True,
    }
    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        info = ydl.extract_info(url, download=True)
    video_id = info["id"]
    audio_path = output_dir / f"{video_id}.wav"
    return audio_path, info.get("title", "unknown"), duration, start, end

for i, url in enumerate(URLS, start=1):
    job_id = f"sample-{i:03d}"
    print("=" * 80)
    print(url, "->", job_id, f"({CLIP_SECONDS}s clip from the middle)")
    try:
        audio_path, title, duration, start, end = extract_middle_clip(url, OUTPUT_DIR, CLIP_SECONDS)
        print(f"Title: {title}")
        print(f"Full video duration: {duration:.1f}s -- clipped [{start:.1f}s, {end:.1f}s]")

        samples, sr = load_mono(audio_path)
        report = analyze_quality(samples, sr)
        decision = select_strategy(report)
        print(report)
        print(decision)

        key = object_key(job_id, "original", audio_path.name)
        upload_file(audio_path, key)
        print(f"Uploaded to Neon Object Storage: {key}")

        url_out = presigned_url(key, expires_in=600)
        print(f"Presigned URL (10 min): {url_out[:80]}...")
    except Exception as e:
        print(f"Failed: {e}")
'''
with open("/content/run_job_example.py", "w") as f:
    f.write(script)

!python /content/run_job_example.py

In [ ]:
# 6. Check sample-004 against the new detectors (background music,
# reverb, compression artifacts) -- this sample came back "clean" on
# the old metrics despite sounding bad by ear.
script = '''
from pathlib import Path
from audiobook_engine.ingestion.youtube import extract_audio, ExtractionError
from audiobook_engine.analysis.io import load_mono
from audiobook_engine.analysis.quality import analyze_quality

URL = "https://youtu.be/mJf-Wm1iJBY"  # sample-004, "clean" under the old metrics
OUTPUT_DIR = Path("/content/samples")

try:
    extracted = extract_audio(URL, OUTPUT_DIR)
    print(f"Title: {extracted.title}")

    samples, sr = load_mono(extracted.audio_path)
    report = analyze_quality(samples, sr)

    print(f"has_background_music:       {report.has_background_music} (confidence={report.background_music_confidence:.3f})")
    print(f"has_reverb:                 {report.has_reverb} (tail_score={report.reverb_tail_score:.3f})")
    print(f"has_compression_artifacts:  {report.has_compression_artifacts} (spectral_cutoff_hz={report.spectral_cutoff_hz:.1f})")
except ExtractionError as e:
    print(f"Extraction failed [{e.kind.value}]: {e}")
'''
with open("/content/check_sample_004.py", "w") as f:
    f.write(script)

!python /content/check_sample_004.py